In [0]:
from pyspark.sql.functions import current_timestamp

calendar_df = (
    spark.table("agentdb.bronze.calendar_raw")
)

calendar_silver = (
    calendar_df
    .select(
        "date",
        "wm_yr_wk",
        "weekday",
        "wday",
        "month",
        "year",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    )
    .dropDuplicates()
    .withColumn(
        "created_at",
        current_timestamp()
    )
)

(
    calendar_silver
    .write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.silver.calendar"
    )
)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

products = (
    spark.table(
        "agentdb.bronze.sales_raw"
    )
    .select(
        "item_id",
        "dept_id",
        "cat_id"
    )
    .dropDuplicates()
)

products = (
    products
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .withColumn(
        "updated_at",
        current_timestamp()
    )
    .withColumn(
        "change_type",
        lit("INSERT")
    )
    .withColumn(
        "is_deleted",
        lit(False)
    )
)

(
    products.write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.silver.product"
    )
)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

stores = (
    spark.table(
        "agentdb.bronze.sales_raw"
    )
    .select(
        "store_id"
    )
    .dropDuplicates()
)

stores = (
    stores
    .withColumnRenamed(
        "store_id",
        "store_code"
    )
    .withColumn(
        "state_id",
        lit("UNKNOWN")
    )
    .withColumn(
        "dc_id",
        lit(None).cast("bigint")
    )
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        lit(None).cast("timestamp")
    )
    .withColumn(
        "is_current",
        lit(True)
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .withColumn(
        "updated_at",
        current_timestamp()
    )
    .withColumn(
        "change_type",
        lit("INSERT")
    )
    .withColumn(
        "is_deleted",
        lit(False)
    )
)

(
    stores.write
    .mode("overwrite")
    .saveAsTable(
        "agentdb.silver.store"
    )
)

In [0]:
from pyspark.sql.functions import current_timestamp, expr

products_df = (
    spark.table("agentdb.silver.product")
    .select(
        "product_id",
        "item_id"
    )
)

stores_df = (
    spark.table("agentdb.silver.store")
    .select(
        "store_id",
        "store_code"
    )
)

prices_df = (
    spark.table(
        "agentdb.bronze.sell_prices_raw"
    )
)

silver_price = (
    prices_df
    .join(
        products_df,
        "item_id",
        "inner"
    )
    .join(
        stores_df,
        prices_df.store_id == stores_df.store_code,
        "inner"
    )
    .select(
        "product_id",
        stores_df.store_id,
        "wm_yr_wk",
        "sell_price"
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .withColumn(
        "load_batch_id",
        expr("uuid()")
    )
)

(
    silver_price.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "agentdb.silver.price"
    )
)

print(
    f"Price Rows: {silver_price.count():,}"
)

silver_price.show(5)

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    expr,
    col
)

sales_raw = spark.table(
    "agentdb.bronze.sales_raw"
)

day_columns = [
    c
    for c in sales_raw.columns
    if c.startswith("d_")
]

stack_expr = ", ".join(
    [
        f"'{c}', {c}"
        for c in day_columns
    ]
)

sales_long = sales_raw.selectExpr(
    "item_id",
    "store_id",
    f"""
    stack(
        {len(day_columns)},
        {stack_expr}
    ) as (
        day_id,
        sales_qty
    )
    """
)

products_df = (
    spark.table(
        "agentdb.silver.product"
    )
    .select(
        "product_id",
        "item_id"
    )
)

stores_df = (
    spark.table(
        "agentdb.silver.store"
    )
    .select(
        "store_id",
        "store_code"
    )
)

calendar_bronze_df = (
    spark.table(
        "agentdb.bronze.calendar_raw"
    )
    .select(
        col("d").alias("day_id"),
        "date"
    )
)

calendar_silver_df = (
    spark.table(
        "agentdb.silver.calendar"
    )
    .select(
        "calendar_id",
        "date"
    )
)

silver_sales = (
    sales_long
    .join(
        products_df,
        "item_id",
        "inner"
    )
    .join(
        stores_df,
        sales_long.store_id
        == stores_df.store_code,
        "inner"
    )
    .join(
        calendar_bronze_df,
        "day_id",
        "inner"
    )
    .join(
        calendar_silver_df,
        "date",
        "inner"
    )
    .select(
        "product_id",
        stores_df.store_id,
        "calendar_id",
        "sales_qty"
    )
    .withColumn(
        "created_at",
        current_timestamp()
    )
    .withColumn(
        "load_batch_id",
        expr("uuid()")
    )
)

(
    silver_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "agentdb.silver.sales"
    )
)

print(
    f"Sales Rows: {silver_sales.count():,}"
)